# Kinematic vs Dynamic Bicycle Model

Comparing the kinematic and dynamic bicycle models from Kong et al., *Kinematic and Dynamic Vehicle Models for Autonomous Driving Control Design*, IEEE IV, 2015.

The **kinematic model** uses 4 states $(x, y, \psi, v)$ and requires only the wheelbase geometry. The **dynamic model** uses 6 states $(\dot{x}, \dot{y}, \psi, \dot{\psi}, X, Y)$ and adds linear tire forces.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pathsim import Simulation, Connection
from pathsim.blocks import Source, Scope

from pathsim_vehicle import (
    KinematicBicycle,
    DynamicBicycle,
    BicycleParameters,
    DynamicBicycleParameters,
    hyundai_azera,
)

## Vehicle Parameters

We use the Hyundai Azera test vehicle from the paper (wheelbase 2.843 m).

In [ ]:
params = hyundai_azera()
print(f"l_f = {params.l_f} m, l_r = {params.l_r} m")
print(f"wheelbase = {params.wheelbase} m")
print(f"m = {params.m} kg, I_z = {params.I_z} kg·m²")

## Step Steer Simulation

Apply a constant steering angle of 5° at a constant speed of 10 m/s, and compare the trajectories predicted by both models.

In [ ]:
delta_f = np.radians(5.0)  # 5° steering
v0 = 10.0                  # m/s
T = 5.0                    # seconds

# --- Kinematic model ---
kin = KinematicBicycle(params, v0=v0)
src_d1 = Source(lambda t: delta_f)
src_a1 = Source(lambda t: 0.0)
sco_k = Scope(labels=['x', 'y', 'psi', 'v'])

sim_k = Simulation(
    [src_d1, src_a1, kin, sco_k],
    [
        Connection(src_d1, kin),
        Connection(src_a1, kin[1]),
        Connection(kin, sco_k),
        Connection(kin[1], sco_k[1]),
        Connection(kin[2], sco_k[2]),
        Connection(kin[3], sco_k[3]),
    ],
    dt=0.01,
)
sim_k.run(T)
t_k, data_k = sco_k.read()

# --- Dynamic model ---
dyn = DynamicBicycle(params, vx0=v0)
src_d2 = Source(lambda t: delta_f)
src_a2 = Source(lambda t: 0.0)
sco_d = Scope(labels=['vx', 'vy', 'psi', 'psi_dot', 'X', 'Y'])

sim_d = Simulation(
    [src_d2, src_a2, dyn, sco_d],
    [
        Connection(src_d2, dyn),
        Connection(src_a2, dyn[1]),
        Connection(dyn, sco_d),
        Connection(dyn[1], sco_d[1]),
        Connection(dyn[2], sco_d[2]),
        Connection(dyn[3], sco_d[3]),
        Connection(dyn[4], sco_d[4]),
        Connection(dyn[5], sco_d[5]),
    ],
    dt=0.01,
)
sim_d.run(T)
t_d, data_d = sco_d.read()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# XY trajectory
ax = axes[0]
ax.plot(data_k[0], data_k[1], label='Kinematic', lw=2)
ax.plot(data_d[4], data_d[5], '--', label='Dynamic', lw=2)
ax.set_xlabel('X [m]')
ax.set_ylabel('Y [m]')
ax.set_title('XY Trajectory Comparison')
ax.legend()
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

# Heading angle
ax = axes[1]
ax.plot(t_k, np.degrees(data_k[2]), label='Kinematic', lw=2)
ax.plot(t_d, np.degrees(data_d[2]), '--', label='Dynamic', lw=2)
ax.set_xlabel('Time [s]')
ax.set_ylabel('Heading ψ [deg]')
ax.set_title('Heading Angle')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

The dynamic model predicts wider turns than the kinematic model at higher speeds due to tire slip (understeering), consistent with the findings in Kong et al. (see Fig. 3 in the paper).